In [403]:
import pandas as pd
import itertools
import plotly.express as px
import plotly.graph_objects as go
import math
import numpy as np
from datetime import datetime
import numpy as np
from scipy.stats import linregress
import matplotlib.pyplot as plt

In [273]:
cset_colors = ["#0B1F41","#003DA6","#B53A6D","#7AC4A5","#F17F4C","#15AFD0","#839DC5"]

In [248]:
#loading the raw data from epoch.ai
all_ai_models = pd.read_csv('../data/ai_models/all_ai_models.csv')

In [249]:
#split out domain list for each model, and add flag if biology is in it
all_ai_models['Domains'] = [d.split(',') if type(d) == str else None for d in all_ai_models['Domain']]
all_ai_models['is_Biology'] = all_ai_models['Domain'].str.contains('Biology') == True
all_ai_models['Model Date'] = [datetime.strptime(d,"%Y-%m-%d").date() if type(d) == str else None for d in all_ai_models['Publication date']]

In [250]:
#List all domains in the data
all_domains = set(itertools.chain.from_iterable([y for y in all_ai_models['Domains'] if y != None]))
all_domains

{'3D modeling',
 'Astronomy',
 'Audio',
 'Biology',
 'Cybersecurity',
 'Driving',
 'Earth science',
 'Games',
 'Image generation',
 'Language',
 'Materials science',
 'Mathematics',
 'Medicine',
 'Multimodal',
 'Other',
 'Psychology',
 'Recommendation',
 'Robotics',
 'Search',
 'Speech',
 'Video',
 'Vision'}

In [251]:
#getting dataset sizes
data_sizes = []
for ds in all_ai_models['Training dataset size (total)']:
    if type(ds) == str:
        sizes = [float(x) for x in ds.split(',')]
        data_sizes.append(max(sizes))
    else:
        if math.isnan(ds):
            data_sizes.append(None)
all_ai_models['Training Dataset Size'] = data_sizes

In [422]:
#getting biodata for plotting
mindate = datetime(2010,1,1).date()
plot_df = all_ai_models[all_ai_models['is_Biology']]
plot_df = plot_df[['Model','Model Date','Training Dataset Size']].dropna()
plot_df = plot_df[plot_df['Training Dataset Size'] > 0]
plot_df = plot_df.sort_values('Model Date')

In [436]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x = plot_df['Model Date'], 
    y = plot_df['Training Dataset Size'],
    mode='markers',
    marker=dict(color=cset_colors[1]),
    customdata = plot_df['Model'],
    hovertemplate='<b>%{customdata}</b><br>%{x}<br><b>Training Dataset Size</b>: %{y:,.2f}<extra></extra>'
))
fig.update_layout(
    title='Training Dataset Size for Biological Models',
    yaxis_type="log",
    template="plotly_white",
    width = 700, height=500,
    xaxis = dict(title='Model Date'),
    yaxis = dict(title='Training Dataset Size')
)
#fig.write_html('bio_data_size.html')
#fig.write_image('bio_data_size.png')
fig.show()

In [430]:
# finding regression
mindate = plot_df['Model Date'].min()
x = np.array([d.days for d in plot_df['Model Date'] - mindate])
x_date = plot_df['Model Date']
y = plot_df['Training Dataset Size']
y_log = np.log(y)

In [431]:
reg = linregress(x,y_log)

In [432]:
y_pred = reg.intercept + reg.slope*x

In [433]:
mse = sum((y_pred - y_log)**2)/(len(y_log)-2)

In [434]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x = plot_df['Model Date'], 
    y = plot_df['Training Dataset Size'],
    mode='markers',
    marker=dict(color=cset_colors[1]),
    customdata = plot_df['Model'],
    hovertemplate='<b>%{customdata}</b><br>%{x}<br><b>Training Dataset Size</b>: %{y}<extra></extra>'
))
fig.add_trace(go.Line(
    x = x_date,
    y = np.exp(y_pred)
))

fig.update_layout(
    title='Training Dataset Size for Biological Models',
    yaxis_type="log",
    template="plotly_white",
    width = 700, height=500,
    xaxis = dict(title='Model Date'),
    yaxis = dict(title='Training Dataset Size'),
    showlegend=False
)
fig.show()